# F1 — EOS Validation: H2STAR vs. NIST

This notebook validates the `h2star.eos` density model (a CoolProp wrapper around the
Leachman et al. 2009 reference equation of state for normal hydrogen) against tabulated
densities from the [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/fluid/).

We compare model and reference density along four isotherms (77, 100, 160, and 298 K)
and render a parity plot: points on the `y = x` diagonal indicate agreement. All
numerical logic lives in the `h2star` package — this notebook only loads data, calls
package functions, and displays the figure.

The relative density errors are asserted below 0.1% in `tests/test_eos.py`.

In [ ]:
from pathlib import Path

import pandas as pd

from h2star import eos, viz

DATA_DIR = Path.cwd().parent / "data" / "validation"
FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(exist_ok=True)

TEMPERATURES_K = [77, 100, 160, 298]

## Load the NIST tables and evaluate the model

Each CSV has provenance comment lines (prefixed `#`) followed by a
`pressure_bar,density_kg_m3` header. Pressure is converted bar→Pa (`* 1e5`) before
calling `eos.density`, which returns kg/m³.

In [ ]:
isotherms = []
for T in TEMPERATURES_K:
    df = pd.read_csv(DATA_DIR / f"nist_h2_{T}K.csv", comment="#")
    P_pa = df["pressure_bar"].to_numpy() * 1e5
    model_density = eos.density(P_pa, T)
    isotherms.append({
        "T": T,
        "nist": df["density_kg_m3"].to_numpy(),
        "model": model_density,
    })

isotherms[0]["model"][:5], isotherms[0]["nist"][:5]

## Parity plot

The figure is produced by `h2star.viz.eos_parity_plot` and saved to
`figures/F1_eos_parity.png`.

In [ ]:
fig, ax = viz.eos_parity_plot(isotherms, savepath=FIG_DIR / "F1_eos_parity.png")
fig

## Result

All four isotherms fall on the `y = x` line, confirming the CoolProp-backed EOS
reproduces NIST reference densities across the pressure range. The quantitative
tolerance (< 0.1% relative error at every point) is enforced by the `validation`-marked
tests in `tests/test_eos.py`.